In [1]:
%%bash
cd /kaggle/working
rm -rf ChestNet

git clone https://github.com/Deep-Learning-Models/ChestNet.git

cd ChestNet
pip install -q -r requirements.txt



Cloning into 'ChestNet'...


In [2]:
%%bash
cd /kaggle/working/ChestNet

# Find the NIH dataset in /kaggle/input/
NIH_PATH=$(find /kaggle/input -name "Data_Entry_2017.csv" -exec dirname {} \; | head -n 1)
echo "Found NIH dataset at: $NIH_PATH"

# Create symlinks
ln -sf "$NIH_PATH/Data_Entry_2017.csv" data/Data_Entry_2017.csv
mkdir -p data/raw
ln -sf "$NIH_PATH"/images_* data/raw/

# Verify
ls -l data/


Found NIH dataset at: /kaggle/input/datasets/nih-chest-xrays/data
total 12
lrwxrwxrwx 1 root root   63 Oct  4 02:37 Data_Entry_2017.csv -> /kaggle/input/datasets/nih-chest-xrays/data/Data_Entry_2017.csv
drwxr-xr-x 2 root root 4096 Oct  4 02:37 raw
-rw-r--r-- 1 root root 2655 Oct  4 02:33 README.md


In [3]:
%%bash
cd /kaggle/working/ChestNet

# 1. Update splits.py to follow directory symlinks
python3 -c '
import re

path = "src/preprocessing/splits.py"
with open(path, "r") as f:
    content = f.read()

# Ensure os is imported
if "import os" not in content:
    content = "import os\n" + content

# Replace rglob with os.walk(followlinks=True)
old_fn = """def _index_images(image_dir: Path) -> dict:
    \"\"\"Map image file name -> full path (Kaggle stores images in images_001 ... images_012).\"\"\"
    return {p.name: str(p) for p in image_dir.rglob("*.png")}"""

new_fn = """def _index_images(image_dir: Path) -> dict:
    \"\"\"Map image file name -> full path (Kaggle stores images in images_001 ... images_012).\"\"\"
    mapping = {}
    for root, _, files in os.walk(str(image_dir), followlinks=True):
        for f in files:
            if f.lower().endswith(".png"):
                mapping[f] = os.path.join(root, f)
    return mapping"""

content = content.replace(old_fn, new_fn)

with open(path, "w") as f:
    f.write(content)
print("Updated splits.py successfully!")
'

# 2. Run the split generation
python -m src.preprocessing.splits


Updated splits.py successfully!
[splits] Saved to data/splits
              train   val  test  total
Atelectasis    1065   205   230   1500
Cardiomegaly   1059   219   255   1533
Effusion       1060   217   231   1508
Infiltration   1155   245   258   1658
Mass           1180   236   246   1662
Nodule         1074   222   229   1525
Pneumonia      1026   205   200   1431
No Finding     1047   219   234   1500
images         5400  1109  1176   7685
patients       3325   713   713   4751
[splits] Leakage check: 0 shared patients, 0 shared images ✅


In [4]:
%%bash
cd /kaggle/working/ChestNet
python -m src.check_pipeline



[1] Shared protocol across models
  ✅ configs/custom_cnn.yaml: locked settings unchanged (overrides: training.learning_rate=0.001)
  ✅ configs/resnet50.yaml: locked settings unchanged (overrides: training.learning_rate=0.0001)
  ✅ configs/efficientnet.yaml: locked settings unchanged (overrides: training.learning_rate=0.0001)
  ✅ configs/vit.yaml: locked settings unchanged (overrides: data.batch_size=16, training.learning_rate=3e-05, training.weight_decay=0.05)

[2-3] Patient-level splits
  ✅ 0 shared patients / 0 shared images across train (5400), val (1109), test (1176)
  ✅ every class has positives in every split

[4-5] tf.data pipeline
  ✅ train batch (32, 224, 224, 3) float32, pixels 0-255, labels multi-hot (32, 8)
  ✅ validation images are identical on every pass (no augmentation, fixed order)
  ✅ validation pass: 7.9 s (fills cache) → 0.29 s (cached)

[6] Class weights (train split only)
       class  train_positives  pos_weight
 Atelectasis             1065       4.070
Cardiome

I0000 00:00:1791081557.197491      87 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1791081557.200892      87 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [5]:
%%bash
cd /kaggle/working/ChestNet
python -m src.train --config configs/vit.yaml


[train] vit: 5400 train / 1109 val images
[train] override data.batch_size: 32 -> 16 (large models (ViT-B/16) need a smaller batch to fit in GPU memory)
[train] override training.learning_rate: 0.0001 -> 3e-05 (pretrained models need a small LR (keep ImageNet features); from-scratch models a larger one)
[train] override training.weight_decay: 0.0001 -> 0.05 (transformers are usually regularised more strongly)
[train] parameters: 85,804,808 total / 85,804,808 trainable
Epoch 1/30
338/338 ━━━━━━━━━━━━━━━━━━━━ 0s 741ms/step - auc: 0.5818 - binary_accuracy: 0.5541 - loss: 1.1127
Epoch 1: val_auc improved from None to 0.73216, saving model to saved_models/vit_best.weights.h5

Epoch 1: finished saving model to saved_models/vit_best.weights.h5
338/338 ━━━━━━━━━━━━━━━━━━━━ 361s 817ms/step - auc: 0.6347 - binary_accuracy: 0.5905 - loss: 1.0680 - val_auc: 0.7322 - val_binary_accuracy: 0.6548 - val_loss: 0.9740 - epoch_time_s: 358.1628
Epoch 2/30
338/338 ━━━━━━━━━━━━━━━━━━━━ 0s 700ms/step - auc: 

I0000 00:00:1791081619.610508     254 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1791081619.612603     254 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1791081714.930490     287 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


In [6]:
%%bash
cd /kaggle/working/ChestNet
python -m src.evaluate --config configs/vit.yaml

74/74 ━━━━━━━━━━━━━━━━━━━━ 20s 267ms/step
       class  support  precision  recall    f1  roc_auc
 Atelectasis      230      0.302   0.813 0.440    0.775
Cardiomegaly      255      0.496   0.831 0.622    0.896
    Effusion      231      0.387   0.593 0.468    0.780
Infiltration      258      0.373   0.519 0.434    0.686
        Mass      246      0.536   0.459 0.495    0.799
      Nodule      229      0.316   0.655 0.427    0.711
   Pneumonia      200      0.337   0.490 0.399    0.692
  No Finding      234      0.342   0.739 0.468    0.765

[evaluate] macro F1 0.4691 | macro ROC-AUC 0.7630 | 16.93 ms/image. Saved to results/vit


I0000 00:00:1791084038.047226     601 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1791084038.049469     601 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5
I0000 00:00:1791084049.828262     632 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
